In [0]:
WITH sales_cte AS (
    SELECT *,

  - Creating the day name
        CASE
            WHEN DAYOFWEEK(transaction_date) = 1 THEN 'Sunday'
            WHEN DAYOFWEEK(transaction_date) = 2 THEN 'Monday'
            WHEN DAYOFWEEK(transaction_date) = 3 THEN 'Tuesday'
            WHEN DAYOFWEEK(transaction_date) = 4 THEN 'Wednesday'
            WHEN DAYOFWEEK(transaction_date) = 5 THEN 'Thursday'
            WHEN DAYOFWEEK(transaction_date) = 6 THEN 'Friday'
            WHEN DAYOFWEEK(transaction_date) = 7 THEN 'Saturday'
        END AS Day_of_week,

 - Creating the time bucket
        CASE
            WHEN HOUR(transaction_time) BETWEEN 7 AND 9 THEN 'Morning'
            WHEN HOUR(transaction_time) BETWEEN 10 AND 12 THEN 'Afternoon'
            WHEN HOUR(transaction_time) BETWEEN 13 AND 15 THEN 'Evening'
            WHEN HOUR(transaction_time) BETWEEN 16 AND 18 THEN 'Night'
            WHEN HOUR(transaction_time) BETWEEN 19 AND 20 THEN 'Closing hours'
            ELSE 'Other'
        END AS Time_bucket
    FROM bright_coffee_shop_analysis
)
SELECT *
FROM sales_cte;

------------------------------------------
- CHECKING TRANSACTIONS AT 6 AM
------------------------------------------

SELECT *
FROM bright_coffee_shop_analysis
WHERE HOUR(transaction_time) = 6
ORDER BY transaction_time;

---------------------------------------
- CHECKING TRANSACTIONS AT 6 PM
---------------------------------------

SELECT *
FROM bright_coffee_shop_analysis
WHERE HOUR(transaction_time) = 18
ORDER BY transaction_time;

---------------------------------------
- CHECKING TRANSACTION HOUR
---------------------------------------

SELECT transaction_time,
       HOUR(transaction_time) AS transaction_hour
FROM bright_coffee_shop_analysis
LIMIT 10;

---------------------------------------------------------
- CREATING DETAILED TRANSACTION TIME BUCKETS
--------------------------------------------------------

SELECT transaction_time,
    HOUR(transaction_time) AS transaction_hour,
    CASE
        WHEN HOUR(transaction_time) BETWEEN 6 AND 8 THEN '06:00–09:00'
        WHEN HOUR(transaction_time) BETWEEN 9 AND 11 THEN '09:00–12:00'
        WHEN HOUR(transaction_time) BETWEEN 12 AND 14 THEN '12:00–15:00'
        WHEN HOUR(transaction_time) BETWEEN 15 AND 17 THEN '15:00–18:00'
        WHEN HOUR(transaction_time) BETWEEN 18 AND 20 THEN '18:00–21:00'
    ELSE 'Other'
    END AS transaction_time_bucket
FROM bright_coffee_shop_analysis;

------------------------------------------------------
- CHECKING HIGHEST AND LOWEST TRANSACTION VALUE
-------------------------------------------------------

SELECT
    ROUND(
        MAX( CAST(transaction_qty AS DOUBLE) *
            CAST(REPLACE(unit_price, ',', '.') AS DOUBLE)), 2 ) AS High_sales_value,
    ROUND(
        MIN( CAST(transaction_qty AS DOUBLE) *
            CAST(REPLACE(unit_price, ',', '.') AS DOUBLE)),2 ) AS Low_sales_value
FROM bright_coffee_shop_analysis;

------------------------------------------
- CREATING SALES VALUE CATEGORY
-----------------------------------------

SELECT transaction_qty, unit_price,
    ROUND(
        CAST(transaction_qty AS DOUBLE)*
        CAST(REPLACE(unit_price, ',', '.') AS DOUBLE), 2 ) AS Total_Amount,
    CASE
        WHEN (
            CAST(transaction_qty AS DOUBLE) *
            CAST(REPLACE(unit_price, ',', '.') AS DOUBLE)) > 10000 THEN 'High'
        WHEN (
            CAST(transaction_qty AS DOUBLE) *
            CAST(REPLACE(unit_price, ',', '.') AS DOUBLE)) BETWEEN 5000 AND 10000 THEN 'Medium'
        ELSE 'Very Low'
    END AS Sales_Value_Category
FROM bright_coffee_shop_analysis;

---------------------------------
- CREATING QUANTITY CATEGORY
----------------------------------

SELECT transaction_qty,
    CASE
        WHEN transaction_qty BETWEEN 1 AND 49 THEN 'Low'
        WHEN transaction_qty BETWEEN 50 AND 100 THEN 'Medium'
        WHEN transaction_qty > 100 THEN 'High'
     ELSE 'Unknown'
    END AS Quantity_Category
FROM bright_coffee_shop_analysis;

-----------------------------------------
- CREATING A CLEANED ANALYSIS TABLE
-------------------------------------------

CREATE OR REPLACE TABLE
bright_coffee_shop_analysis.bright_coffee_shop_analysis_cleaned AS

SELECT transaction_id,
    -- Date
    CAST(transaction_date AS DATE) AS transaction_date,
    -- Day name
    CASE
        WHEN DAYOFWEEK(CAST(transaction_date AS DATE)) = 1 THEN 'Sunday'
        WHEN DAYOFWEEK(CAST(transaction_date AS DATE)) = 2 THEN 'Monday'
        WHEN DAYOFWEEK(CAST(transaction_date AS DATE)) = 3 THEN 'Tuesday'
        WHEN DAYOFWEEK(CAST(transaction_date AS DATE)) = 4 THEN 'Wednesday'
        WHEN DAYOFWEEK(CAST(transaction_date AS DATE)) = 5 THEN 'Thursday'
        WHEN DAYOFWEEK(CAST(transaction_date AS DATE)) = 6 THEN 'Friday'
        WHEN DAYOFWEEK(CAST(transaction_date AS DATE)) = 7 THEN 'Saturday'
    END AS day_name,
    -- Day number
    DAYOFWEEK(
        CAST(transaction_date AS DATE) ) AS day_number,
    -- Month name
    DATE_FORMAT(
        CAST(transaction_date AS DATE), 'MMMM' ) AS month_name,
    -- Month number
    MONTH( CAST(transaction_date AS DATE) ) AS month_number,
    -- Year
    YEAR( CAST(transaction_date AS DATE) ) AS year,
    -- Time of day
    CASE
        WHEN HOUR(transaction_time) BETWEEN 6 AND 8 THEN '06:00–09:00'
        WHEN HOUR(transaction_time) BETWEEN 9 AND 11 THEN '09:00–12:00'
        WHEN HOUR(transaction_time) BETWEEN 12 AND 14 THEN '12:00–15:00'
        WHEN HOUR(transaction_time) BETWEEN 15 AND 17 THEN '15:00–18:00'
        WHEN HOUR(transaction_time) BETWEEN 18 AND 20 THEN '18:00–21:00'
    ELSE 'Other'
 END AS time_of_day,
    -- Time bucket
    CASE
        WHEN HOUR(transaction_time) BETWEEN 7 AND 9 THEN 'Morning'
        WHEN HOUR(transaction_time) BETWEEN 10 AND 12 THEN 'Afternoon'
        WHEN HOUR(transaction_time) BETWEEN 13 AND 15 THEN 'Evening'
        WHEN HOUR(transaction_time) BETWEEN 16 AND 18 THEN 'Night'
        WHEN HOUR(transaction_time) BETWEEN 19 AND 20 THEN 'Closing hours'
   ELSE 'Other'
END AS time_bucket,
    -- Transaction quantity
    CAST(transaction_qty AS DOUBLE) AS transaction_qty,
    -- Store information
    store_id,
    store_location,
    -- Product information
    product_id,
    product_category,
    product_type,
    product_detail,
    -- Unit price
    CAST(
        REPLACE( CAST(unit_price AS STRING), ',', '.' ) AS DECIMAL(10,2) ) AS unit_price,
    -- Total transaction amount
    CAST(transaction_qty AS DOUBLE) *
    CAST( REPLACE( CAST(unit_price AS STRING), ',', '.' ) AS DECIMAL(10,2) ) AS total_amount
FROM bright_coffee_shop_analysis;

SELECT *
FROM bright_coffee_shop_analysis_cleaned;